In [ ]:
"""
1. Data Cleaning

Given a DataFrame like:

id	name	age	city
1	Alice	25	NY
2	Bob	null	LA
2	Bob	null	LA
3	null	40	NY

Ask the candidate to:

Remove duplicates
Fill missing ages with the average age
Replace missing names with "Unknown"
Filter people over 30
"""
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName("practice1").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("city", StringType(), True)
])

                # json file was a multiline array
                # A nested json file would require exploding
df = spark.read.option("multiline", "true").json("num1.json", schema=schema1)

# Remove duplicates
df_unique = df.dropDuplicates()

# Fill missing age with average age
avg_age = df_unique.select(F.avg("age")).first()[0]
df_unique = df_unique.withColumn(
    "age",
    F.when(F.col("age").isNull(), avg_age).otherwise(F.col("age"))
)

# replace missing names wih unknown
df_unique = df_unique.fillna({"name":"Unknown"})

# filter people over 30
df_unique = df_unique.filter(F.col("age") > 30)

# show the results:
print("\nOriginal:")
df.show()

print("\nAfter Cleaning:")
df_unique.show()

spark.stop()


Original:
+---+-----+----+----+
| id| name| age|city|
+---+-----+----+----+
|  1|Alice|  25|  NY|
|  2|  Bob|NULL|  LA|
|  2|  Bob|NULL|  LA|
|  3| NULL|  40|  NY|
+---+-----+----+----+


After Cleaning:
+---+-------+----+----+
| id|   name| age|city|
+---+-------+----+----+
|  2|    Bob|32.5|  LA|
|  3|Unknown|40.0|  NY|
+---+-------+----+----+



In [10]:
"""
2. Group By and Aggregation

Sales data:

customer	product	amount
A	Phone	500
A	Laptop	1000
B	Phone	700

Questions:

Total spending per customer
Average purchase amount
Highest spending customer
"""
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = SparkSession.builder.appName("practice2").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("customer", StringType(), True),
    StructField("product", StringType()),
    StructField("amount", IntegerType())
])

df = spark.read.csv("num2.csv", schema=schema1, header=True)

# get total spending by customer
customer_total_spending_df = df.groupBy("customer").agg(F.sum("amount").alias("total_spent"))

# Average purchase amount per customer
customer_avg_spending_df = df.groupBy("customer").agg(F.avg("amount").alias("avg_spent"))

# Biggest spender
biggest_spender_df = df.groupBy("customer") \
                        .agg(F.sum("amount").alias("Highest Spending")) \
                        .orderBy(F.desc("Highest Spending")) \
                        .limit(1)


# print results
print("\nOriginal:")
df.show()

print("\nTotal Spent Per Customer:")
customer_total_spending_df.show()

print("\nAverage Spent Per Customer:")
customer_avg_spending_df.show()

print("\nCustomer who spent the most:")
biggest_spender_df.show()

spark.stop()


Original:
+--------+----------+------+
|customer|   product|amount|
+--------+----------+------+
|       A|     Phone|   500|
|       A|    Laptop|  1000|
|       B|     Phone|   700|
|       B|Headphones|   150|
|       C|    Laptop|  1200|
|       C|     Mouse|    50|
|       A|     Mouse|    25|
|       B|    Laptop|   900|
|       C|     Phone|   650|
+--------+----------+------+


Total Spent Per Customer:
+--------+-----------+
|customer|total_spent|
+--------+-----------+
|       B|       1750|
|       C|       1900|
|       A|       1525|
+--------+-----------+


Average Spent Per Customer:
+--------+-----------------+
|customer|        avg_spent|
+--------+-----------------+
|       B|583.3333333333334|
|       C|633.3333333333334|
|       A|508.3333333333333|
+--------+-----------------+


Customer who spent the most:
+--------+----------------+
|customer|Highest Spending|
+--------+----------------+
|       C|            1900|
+--------+----------------+



In [ ]:
"""
3. Top N per Group

Example:

department	employee	salary
IT	John	100
IT	Mary	200
IT	Alex	150

Return the highest-paid employee in each department.

Expected solution:

Use a Window function.
""" 
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructField, StructType, StringType, IntegerType
from pyspark.sql.window import Window

spark = spark.builder.appName("problem3").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("department", StringType(), True),
    StructField("employee", StringType(), True),
    StructField("salary", IntegerType(), True)
])

df = spark.read.csv("num3.csv", schema=schema1, header=False)

# give each department a separate window
depWindow = Window.partitionBy("department").orderBy(F.desc("salary"))

# rank the highest spenders
df = df.withColumn(
    "rank",
    F.row_number().over(depWindow)
)

# find highest paid employee for each department
highest_paid = df.filter(F.col("rank") == 1).drop("rank")


# print results
print("\nOriginal:")
df.drop("rank").show()

print("\nHighest Paid Employee per Department:")
highest_paid.show()

spark.stop()


Original:
+----------+--------+------+
|department|employee|salary|
+----------+--------+------+
|        IT|    John|   100|
|        IT|    Mary|   200|
|        IT|    Alex|   150|
|        HR|   Susan|   120|
|        HR|     Bob|   180|
|        HR|   Alice|   160|
|     Sales|     Tom|   250|
|     Sales|    Jane|   300|
|     Sales|    Mike|   275|
|   Finance|    Emma|   220|
|   Finance|   David|   210|
|   Finance|  Olivia|   260|
+----------+--------+------+


Highest Paid Employee per Department:
+----------+--------+------+
|department|employee|salary|
+----------+--------+------+
|   Finance|  Olivia|   260|
|        HR|     Bob|   180|
|        IT|    Mary|   200|
|     Sales|    Jane|   300|
+----------+--------+------+



In [23]:
""" 
4. Joins

Customers:
id	name
1	Alice
2	Bob

Orders:
customer_id	amount
1	100
1	50
3	25

Questions:

Inner join
Left join
Customers with no orders
Total spend per customer
"""
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName("problem4").master("local[*]").getOrCreate()

schemaOrders = StructType([
    StructField("customer_id", StringType(), True),
    StructField("amount", IntegerType(), True)
])

schemaCustomers = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True)
])

# Create DataFrames
orders = spark.read.csv("num4_orders.csv", schema=schemaOrders, header=False)
customers = spark.read.csv("num4_customers.csv", schema=schemaCustomers, header=False)

# Inner join: only rows that matched in both
inner_joined = customers.join(
    orders,
    on=customers["id"] == orders["customer_id"],
    how="inner"
)

# Left Join: all rows from left, plus matches from right
left_joined = customers.join(
    orders,
    on=customers["id"] == orders["customer_id"],
    how="left"
)

# Customers with no orders (anti left/right join)
customers_no_orders = customers.join(
    orders,
    on=customers["id"] == orders["customer_id"],
    how="left"
)
customers_no_orders = customers_no_orders.filter(F.col("amount").isNull())

# total spent per customer
total_spending = left_joined.fillna({"amount": 0})
total_spending = total_spending.groupBy("name") \
                .agg(F.sum("amount").alias("Total Spent")) \
                .orderBy(F.desc("Total Spent"))
    

# print results
print("Inner Join:")
inner_joined.show()

print("Left Join:")
left_joined.show()

print("Customers with no orders:")
customers_no_orders.select("id", "name").show()

print("Total Spent Per Customer:")
total_spending.show()


spark.stop()

Inner Join:
+---+-------+-----------+------+
| id|   name|customer_id|amount|
+---+-------+-----------+------+
|  1|  Alice|          1|    50|
|  1|  Alice|          1|   100|
|  3|Charlie|          3|    75|
|  3|Charlie|          3|    25|
+---+-------+-----------+------+

Left Join:
+---+-------+-----------+------+
| id|   name|customer_id|amount|
+---+-------+-----------+------+
|  1|  Alice|          1|    50|
|  1|  Alice|          1|   100|
|  2|    Bob|       NULL|  NULL|
|  3|Charlie|          3|    75|
|  3|Charlie|          3|    25|
|  4|  Diana|       NULL|  NULL|
+---+-------+-----------+------+

Customers with no orders:
+---+-----+
| id| name|
+---+-----+
|  2|  Bob|
|  4|Diana|
+---+-----+

Total Spent Per Customer:
+-------+-----------+
|   name|Total Spent|
+-------+-----------+
|  Alice|        150|
|Charlie|        100|
|  Diana|          0|
|    Bob|          0|
+-------+-----------+



In [33]:
""" 
5. Find Duplicate Records

Dataset:

email
-----
a@test.com
b@test.com
a@test.com
c@test.com

Tasks:

Find duplicate emails
Count occurrences
Keep only unique rows
""" 
from pyspark.sql import SparkSession, functions as F 

spark = SparkSession.builder.appName("example5").master("local[*]").getOrCreate()

# read the text file as a csv to be able to use sep="\n"
df = spark.read.csv("num5.txt", sep="\n", header=True)

# count duplicates by creating a count column that will get summed
count_emails = df.withColumn("count", F.lit(1)) \
    .groupBy("email").agg(F.sum("count").alias("count"))

# emails with duplicates
count_duplicate_emails = count_emails.filter(F.col("count") > 1)

# only the emails (no count)
duplicate_emails = count_duplicate_emails.drop("count")

# find unique emails
df_unique = df.dropDuplicates()

# print values
print("Original:")
df.show()

print("Duplicate Emails:")
duplicate_emails.show()

print("Count of Duplicate Emails:")
count_duplicate_emails.show()

print("No Duplicate Emails:")
df_unique.show()

spark.stop()

Original:
+----------+
|     email|
+----------+
|a@test.com|
|b@test.com|
|a@test.com|
|c@test.com|
+----------+

Duplicate Emails:
+----------+
|     email|
+----------+
|a@test.com|
+----------+

Count of Duplicate Emails:
+----------+-----+
|     email|count|
+----------+-----+
|a@test.com|    2|
+----------+-----+

No Duplicate Emails:
+----------+
|     email|
+----------+
|b@test.com|
|a@test.com|
|c@test.com|
+----------+



In [ ]:
""" 
6. Window Function Challenge

Transactions:
user	date	amount
A	Jan1	10
A	Jan2	20
A	Jan3	15

Questions:

Running total
Previous transaction
Difference from previous transaction
Rolling 7-day average
""" 
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("problem6").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("user", StringType(), True),
    StructField("date", DateType(), True),  # NOTE: DateType() exists for 2025-01-01
    StructField("amount", IntegerType(), True)
])

df = spark.read.csv("num6.csv", schema=schema1, header=True)

# Get separate windows for each user
dfWindow = Window.partitionBy("user").orderBy("date")

# get a running total of the amount
RunningTotal = df.withColumn(
    "running_total",
    F.sum("amount").over(dfWindow)
)

# find difference from previous transaction in the window
dif_from_prev = df.withColumn(
    "difference",
    F.col("amount") - F.lag("amount").over(dfWindow)  
)

# 7 day rolling window
# use days since 1970-01-01
prev_7_days = (
    Window.partitionBy("user")
    .orderBy(F.datediff(F.col("date"), F.lit("1970-01-01")))
    .rangeBetween(-7, 0)
)

# find rolling 7 day average
rolling_7_avg = df.withColumn(
    "7_day_avg",
    F.avg("amount").over(prev_7_days)
)

print("Original:")
df.show()

print("Running Total:")
RunningTotal.show()

print("Difference from previous transaction:")
dif_from_prev.show()

print("Rolling window over 7 days:")
rolling_7_avg.show()

spark.stop()

Original:
+----+----------+------+
|user|      date|amount|
+----+----------+------+
|   A|2025-01-01|    10|
|   A|2025-01-02|    20|
|   A|2025-01-03|    15|
|   A|2025-01-05|    30|
|   A|2025-01-08|    25|
|   B|2025-01-01|    50|
|   B|2025-01-03|    40|
|   B|2025-01-04|    60|
|   B|2025-01-10|    80|
+----+----------+------+

Running Total:
+----+----------+------+-------------+
|user|      date|amount|running_total|
+----+----------+------+-------------+
|   A|2025-01-01|    10|           10|
|   A|2025-01-02|    20|           30|
|   A|2025-01-03|    15|           45|
|   A|2025-01-05|    30|           75|
|   A|2025-01-08|    25|          100|
|   B|2025-01-01|    50|           50|
|   B|2025-01-03|    40|           90|
|   B|2025-01-04|    60|          150|
|   B|2025-01-10|    80|          230|
+----+----------+------+-------------+

Difference from previous transaction:
+----+----------+------+----------+
|user|      date|amount|difference|
+----+----------+------+-------

In [17]:
""" 
7. Sessionization

Given clickstream events:
user
timestamp

Create sessions where a new session starts if the gap exceeds 30 minutes.
"""
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("problem7").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("user", StringType(), True),
    StructField("timestamp", TimestampType(), True)
])

df = spark.read.csv("num7.csv", schema=schema1, header=True)

# partition separate windows for each user
df_window = Window.partitionBy("user").orderBy("timestamp")

# use windows to figure out differences between timestamps
# can't cast timestamp to long because of NULLs, use coalesce()
# divide by 60 to convert seconds to minutes
df_with_time_dif = df.withColumn(
    "difference",
    (F.col("timestamp").cast("long") - F.lag("timestamp").over(df_window).cast("long")) / 60
).fillna(0)

# mark when new sessions are needed
df_sessions = df_with_time_dif.withColumn(
    "session",
    F.when((F.col("difference") > 30), 1).otherwise(0)
)

# get a running total of the sessions
# This will assign the correct session number
df_sessions = df_sessions.withColumn(
    "session",
    F.sum("session").over(df_window)
)

print("Original:")
df.show()

print("Time stamps with differences:")
df_sessions.show()

spark.stop()

Original:
+----+-------------------+
|user|          timestamp|
+----+-------------------+
|   A|2025-01-01 09:00:00|
|   A|2025-01-01 09:10:00|
|   A|2025-01-01 09:25:00|
|   A|2025-01-01 10:05:00|
|   A|2025-01-01 10:15:00|
|   B|2025-01-01 12:00:00|
|   B|2025-01-01 12:20:00|
|   B|2025-01-01 13:00:00|
|   B|2025-01-01 13:10:00|
|   B|2025-01-01 13:41:00|
|   C|2025-01-01 08:00:00|
|   C|2025-01-01 09:00:00|
|   C|2025-01-01 09:40:00|
+----+-------------------+

Time stamps with differences:
+----+-------------------+----------+-------+
|user|          timestamp|difference|session|
+----+-------------------+----------+-------+
|   A|2025-01-01 09:00:00|       0.0|      0|
|   A|2025-01-01 09:10:00|      10.0|      0|
|   A|2025-01-01 09:25:00|      15.0|      0|
|   A|2025-01-01 10:05:00|      40.0|      1|
|   A|2025-01-01 10:15:00|      10.0|      1|
|   B|2025-01-01 12:00:00|       0.0|      0|
|   B|2025-01-01 12:20:00|      20.0|      0|
|   B|2025-01-01 13:00:00|      40.0|   

In [ ]:
""" 
8. Pivot Challenge

Input

Month	Product	Sales
Jan	A	10
Jan	B	15
Feb	A	12

Output

Month	A	B
Jan	10	15
Feb	12	null
"""
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName("problem8").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("Month", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Sales", IntegerType(), True)
])

df = spark.read.csv("num8.csv", schema=schema1, header=True)

pivoted = df.groupBy("Month").pivot("Product").sum("Sales")

print("Original:")
df.show()

print("Pivoted:")
pivoted.show()

spark.stop()

Original:
+-----+-------+-----+
|Month|Product|Sales|
+-----+-------+-----+
|  Jan|      A|   10|
|  Jan|      B|   15|
|  Feb|      A|   12|
|  Feb|      B|   18|
|  Mar|      A|   20|
|  Mar|      C|   25|
|  Apr|      B|   30|
|  Apr|      C|   35|
|  May|      A|   22|
|  May|      B|   28|
|  May|      C|   32|
|  Jun|      A|   18|
|  Jun|      C|   40|
+-----+-------+-----+

Pivoted:
+-----+----+----+----+
|Month|   A|   B|   C|
+-----+----+----+----+
|  May|  22|  28|  32|
|  Jun|  18|NULL|  40|
|  Feb|  12|  18|NULL|
|  Mar|  20|NULL|  25|
|  Jan|  10|  15|NULL|
|  Apr|NULL|  30|  35|
+-----+----+----+----+



In [21]:
""" 
9. Explode Nested Data

Input:

[
  ("Alice", ["Math", "Science"]),
  ("Bob", ["History"])
]

Output:

name	subject
Alice	Math
Alice	Science
Bob	History
"""
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.appName("problem9").master("local[*]").getOrCreate()

schema1 = StructType([
    StructField("name", StringType(), False),
    StructField("subjects", StringType(), True)
])

df = spark.read.csv("num9.csv", schema=schema1, header=True)

df_updated = df.withColumn(
    "subjects",
    F.split(F.col("subjects"), "\\|")
)

df_updated = df_updated.select(
    "name",
    F.explode("subjects").alias("subject")
)

print("Original:")
df.show()

print("Exploded:")
df_updated.show()

Original:
+-------+-----------------+
|   name|         subjects|
+-------+-----------------+
|  Alice|     Math|Science|
|    Bob|          History|
|Charlie|English|Art|Music|
|  David|             Math|
|    Eva|  Science|History|
+-------+-----------------+

Exploded:
+-------+-------+
|   name|subject|
+-------+-------+
|  Alice|   Math|
|  Alice|Science|
|    Bob|History|
|Charlie|English|
|Charlie|    Art|
|Charlie|  Music|
|  David|   Math|
|    Eva|Science|
|    Eva|History|
+-------+-------+



In [ ]:
""" 
10. JSON Parsing

Input:

{
  "customer":{
      "id":1,
      "city":"NY"
  }
}

Tasks:

Parse JSON
Flatten nested columns
Select nested fields
"""
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("problem10").master("local[*]").getOrCreate()

df = spark.read.option("multiLine", True).json("num10.json")

df_flatten = df.select("customer.*")

print("\nOriginal DataFrame:")
df.show()

print("Original schema:")
df.printSchema()

print("Selecting nested fields:")
df.select("customer.id", "customer.city").show()

print("Flattened DataFrame:")
df_flatten.show()

print("Flattened schema:")
df_flatten.printSchema()

spark.stop()


Original DataFrame:
+----------------+
|        customer|
+----------------+
|         {NY, 1}|
|    {Chicago, 2}|
|{Los Angeles, 3}|
|    {Seattle, 4}|
|     {Boston, 5}|
+----------------+

Original schema:
root
 |-- customer: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- id: long (nullable = true)

Select nested fields
+---+-----------+
| id|       city|
+---+-----------+
|  1|         NY|
|  2|    Chicago|
|  3|Los Angeles|
|  4|    Seattle|
|  5|     Boston|
+---+-----------+

Flattened DataFrame:
+-----------+---+
|       city| id|
+-----------+---+
|         NY|  1|
|    Chicago|  2|
|Los Angeles|  3|
|    Seattle|  4|
|     Boston|  5|
+-----------+---+

Flattened schema:
root
 |-- city: string (nullable = true)
 |-- id: long (nullable = true)



In [ ]:
""" 
End-to-End ETL Challenge (Excellent 60-minute Interview)

Orders:
order_id
customer_id
date
amount

Customers:
customer_id
city
segment

Requirements:
- Clean missing values
- Remove duplicates
- Join datasets
- Calculate:
    - total sales by city
    - average order value by segment
    - top 3 customers by spending
- Write the result partitioned by city
""" 
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, FloatType

spark = SparkSession.builder.appName("challenge").master("local[*]").getOrCreate()

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True)
])

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("date", DateType(), True),
    StructField("amount", FloatType(), True)
])

customers = spark.read.csv("challenge_customers.csv", schema=customers_schema, header=True)
orders = spark.read.csv("challenge_orders.csv", schema=orders_schema, header=True)

# clean customers
customers_clean = customers.fillna({
    "city": "Unknown",
    "segment": "Unknown"
})

# clean orders
orders_clean = orders.fillna({
    "amount": 0
})

# remove duplicates
customers_clean = customers_clean.dropDuplicates()
orders_clean = orders_clean.dropDuplicates()

# inner join the cleaned/deduplicated customers and orders DataFrames
joined_df = orders_clean.join(
    customers_clean,
    on=orders_clean.customer_id == customers_clean.customer_id,
    how="inner"
).drop(customers.customer_id).cache()

# get total sales for each city
sales_by_city = joined_df.groupBy("city") \
    .agg(F.round(F.sum("amount"), 2).alias("total_sales")) \
    .orderBy(F.desc("total_sales"))

# get the average order value by segment
segment_avg_order = joined_df.groupBy("segment") \
    .agg(F.round(F.avg("amount"), 2).alias("avg_segment_sales")) \
    .orderBy(F.desc("avg_segment_sales"))

# get the top 3 highest spenders
top_3_customers = joined_df.groupBy("customer_id") \
    .agg(F.round(F.sum("amount"), 2).alias("total_spent")) \
    .orderBy(F.desc("total_spent")) \
    .limit(3)


print("Original customers DataFrame:")
customers.show()
print("Original orders DataFrame:")
orders.show()

print("\nJoined DataFrame:")
joined_df.show()

print("Total Sales By City:")
sales_by_city.show()

print("Average Order Value by Segment:")
segment_avg_order.show()

print("Top 3 Highest Spending Customers:")
top_3_customers.show()


# write results partitioned by city
joined_df.write.partitionBy("city").parquet("challenge_output", mode="overwrite")

spark.stop()

Original customers DataFrame:
+-----------+--------+-----------+
|customer_id|    city|    segment|
+-----------+--------+-----------+
|          1|New York|   Consumer|
|          2| Chicago|  Corporate|
|          3|New York|  Corporate|
|          4| Seattle|   Consumer|
|          5| Chicago|Home Office|
|          6|  Boston|  Corporate|
|          7| Seattle|   Consumer|
|          8|New York|Home Office|
|          9|  Boston|   Consumer|
|         10| Chicago|  Corporate|
|         11| Seattle|       NULL|
|         12|New York|   Consumer|
|         13|    NULL|  Corporate|
+-----------+--------+-----------+

Original orders DataFrame:
+--------+-----------+----------+------+
|order_id|customer_id|      date|amount|
+--------+-----------+----------+------+
|    1001|        001|2025-01-02| 120.5|
|    1002|        002|2025-01-02|  85.0|
|    1003|        003|2025-01-03|240.75|
|    1004|        001|2025-01-05|  60.0|
|    1005|        004|2025-01-06|150.25|
|    1006|        0

26/07/10 11:22:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
